# 실습: Gemini API 기본 호출

이 노트북은 `main.py`의 내용을 셀 단위 강의자료로 바꾼 버전입니다.

## 학습 목표

- `.env`에서 `GEMINI_API_KEY`를 안전하게 읽는다
- `google-genai` SDK로 첫 Gemini API 요청을 보낸다
- `response.text`, `usage_metadata`, `finish_reason`을 확인한다
- 시스템 지시문, 토큰 계산, 에러 처리 흐름을 셀별로 익힌다
- API 요청 수를 보며 수업 중 호출량을 조절한다

## 수업 진행 방식

- 위에서 아래로 한 셀씩 실행합니다.
- API 요청이 발생하는 셀은 `[API 요청 n]` 형태로 요청 번호를 출력합니다.
- 429 `RESOURCE_EXHAUSTED`가 나오면 할당량 또는 짧은 시간 안의 요청량 문제일 수 있습니다.
- 수업 중에는 1~4번 요청까지만 실행해도 기본 개념을 충분히 확인할 수 있습니다.

## 0. 준비 확인

먼저 `.env` 파일에 아래 값이 들어 있어야 합니다.

```text
GEMINI_API_KEY=발급받은_API_KEY
```

`GEMINI_MODEL`을 지정하지 않으면 기본값으로 `gemini-3.5-flash`를 사용합니다.

In [ ]:
import os
import time

from dotenv import load_dotenv
from google import genai
from google.genai import errors, types

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise RuntimeError(".env 파일에 GEMINI_API_KEY=... 값을 넣어주세요.")

MODEL = os.getenv("GEMINI_MODEL", "gemini-3.5-flash")
REQUEST_DELAY_SECONDS = float(os.getenv("GEMINI_REQUEST_DELAY_SECONDS", "1.0"))

client = genai.Client(api_key=api_key)
api_request_count = 0

print(f"MODEL = {MODEL}")
print(f"REQUEST_DELAY_SECONDS = {REQUEST_DELAY_SECONDS}")

## 1. 공통 도우미 함수

여러 셀에서 같은 방식으로 API를 호출하기 위해 작은 함수를 먼저 만들어 둡니다.

- 요청 번호를 출력합니다.
- 요청 사이에 짧은 간격을 둡니다.
- Gemini API 오류가 나면 `code`, `status`, `message`를 보기 쉽게 출력합니다.

In [ ]:
def call_gemini(label, request_func):
    """Gemini API 요청을 보낼 때 요청 번호와 에러를 함께 관리한다."""
    global api_request_count
    api_request_count += 1

    print(f"\n[API 요청 {api_request_count}] {label}")
    print("-" * 50)

    if api_request_count > 1:
        time.sleep(REQUEST_DELAY_SECONDS)

    try:
        return request_func()
    except errors.APIError as exc:
        message = exc.message or str(exc)
        print(f"APIError [{exc.code} {exc.status}]: {message[:300]}")
        return None
    except Exception as exc:
        print(f"Unexpected error [{type(exc).__name__}]: {str(exc)[:300]}")
        return None


def show_response(title, response, max_chars=600):
    """응답 텍스트를 너무 길지 않게 출력한다."""
    if response is None:
        print("응답이 없습니다.")
        return

    text = response.text or ""
    print(f"[{title}]")
    print(text[:max_chars])


def show_usage(response):
    """응답 객체에서 토큰 사용량과 종료 이유를 꺼내 출력한다."""
    if response is None:
        return

    usage = response.usage_metadata
    candidate = response.candidates[0] if response.candidates else None

    print("\n[토큰 사용량]")
    print(f"입력 토큰: {usage.prompt_token_count}")
    print(f"출력 토큰: {usage.candidates_token_count}")
    print(f"총 토큰:   {usage.total_token_count}")

    if candidate:
        print(f"중단 이유: {candidate.finish_reason}")

## 2. 첫 번째 Gemini API 호출

`client.models.generate_content()`는 가장 기본적인 텍스트 생성 메서드입니다.

- `model`: 사용할 모델 이름
- `contents`: 모델에게 보낼 질문 또는 지시문
- `config`: 출력 길이, 창의성 같은 옵션

In [ ]:
response = call_gemini(
    "첫 번째 Gemini API 호출",
    lambda: client.models.generate_content(
        model=MODEL,
        contents="안녕하세요! 한 문장으로 자기소개를 해주세요.",
        config=types.GenerateContentConfig(max_output_tokens=120),
    ),
)

show_response("Gemini 응답", response)

## 3. 응답 구조 분석

Gemini 응답에서 가장 자주 보는 값은 `response.text`입니다.

하지만 서비스 개발에서는 토큰 사용량과 종료 이유도 중요합니다.

- `usage_metadata.prompt_token_count`: 입력 토큰 수
- `usage_metadata.candidates_token_count`: 출력 토큰 수
- `usage_metadata.total_token_count`: 총 토큰 수
- `candidates[0].finish_reason`: 응답이 끝난 이유

In [ ]:
response = call_gemini(
    "응답 구조 분석",
    lambda: client.models.generate_content(
        model=MODEL,
        contents="Python의 장점 3가지를 짧은 글머리표로 나열해주세요.",
        config=types.GenerateContentConfig(max_output_tokens=180),
    ),
)

show_response("응답 텍스트", response)
show_usage(response)

## 4. 시스템 지시문

`system_instruction`은 모델에게 역할과 답변 방식을 미리 알려주는 설정입니다.

아래 셀에서 `role`과 `question`을 바꿔 보면서 답변 톤이 어떻게 달라지는지 확인합니다.

In [ ]:
role = "친절한 한국어 선생님"
question = "영어 문장 'The quick brown fox'를 초보자에게 설명해주세요."

response = call_gemini(
    "시스템 지시문 실습",
    lambda: client.models.generate_content(
        model=MODEL,
        contents=question,
        config=types.GenerateContentConfig(
            system_instruction=f"당신은 {role}입니다. 항상 그 역할에 맞게 답변하세요.",
            max_output_tokens=220,
        ),
    ),
)

print(f"역할: {role}")
print(f"질문: {question}")
show_response("답변", response)

### 역할 바꿔보기

수업 중 시간이 있으면 아래 예시 중 하나를 골라 위 셀의 `role`, `question`에 넣어 실행해 봅니다.

In [ ]:
role_examples = [
    ("친절한 한국어 선생님", "영어 문장 'The quick brown fox'를 설명해주세요."),
    ("엄격한 코드 리뷰어", "아래 코드를 검토해주세요:\ndef add(a, b):\n  return a+b"),
    ("5살 아이에게 설명하는 과학자", "블랙홀이 뭔가요?"),
]

for index, (example_role, example_question) in enumerate(role_examples, start=1):
    print(f"{index}. {example_role} -> {example_question[:40]}")

## 5. 토큰 사전 카운팅

토큰은 AI 모델이 텍스트를 나누어 읽는 단위입니다.

긴 입력을 보내기 전에 `count_tokens()`로 대략적인 입력 크기를 확인할 수 있습니다.

In [ ]:
texts = [
    "Hello",
    "안녕하세요",
    "FastAPI는 Python 기반의 고성능 웹 프레임워크로, 자동 문서화와 타입 안전성을 제공합니다.",
    "Explain the difference between synchronous and asynchronous programming in Python with examples.",
]

for text in texts:
    token_response = call_gemini(
        f"토큰 계산: {text[:20]}",
        lambda text=text: client.models.count_tokens(
            model=MODEL,
            contents=text,
        ),
    )

    if token_response:
        print(f"{token_response.total_tokens:4d} tokens | {text[:70]}")

## 6. 에러 처리

API를 쓰는 코드는 실패 상황도 함께 다뤄야 합니다.

아래 셀은 일부러 잘못된 모델명을 사용해서 `APIError`가 어떻게 출력되는지 확인합니다.

In [ ]:
bad_response = call_gemini(
    "잘못된 모델명으로 에러 확인",
    lambda: client.models.generate_content(
        model="gemini-nonexistent-model",
        contents="안녕",
    ),
)

print("bad_response =", bad_response)

### 자주 만나는 오류

| 코드 | 의미 | 확인할 것 |
|------|------|-----------|
| 400 | 요청 형식 오류 | 모델명, 파라미터, 입력 형식 |
| 403 | 인증 또는 권한 문제 | API 키, 프로젝트 권한 |
| 429 | 요청 한도 초과 | 잠시 대기, 요청 수 줄이기 |
| 500 | 서버 내부 오류 | 잠시 후 재시도 |

## 7. 사용 가능한 모델 목록 조회

현재 계정에서 사용할 수 있는 모델을 확인합니다.

모델 목록은 시간이 지나면서 바뀔 수 있으므로, 수업 자료의 표보다 실제 API 결과를 우선 확인합니다.

In [ ]:
models_response = call_gemini(
    "모델 목록 조회",
    lambda: list(client.models.list()),
)

if models_response:
    gemini_models = [m for m in models_response if "gemini" in m.name.lower()]

    for model in gemini_models[:10]:
        name = model.name.split("/")[-1] if "/" in model.name else model.name
        print(name)

    if len(gemini_models) > 10:
        print(f"... 외 {len(gemini_models) - 10}개")

## 8. `main.py`와 비교하기

`main.py`에서는 아래 함수들이 한 번에 실행됩니다.

| `main.py` 함수 | 노트북 위치 |
|----------------|-------------|
| `hello_gemini()` | 2. 첫 번째 Gemini API 호출 |
| `explore_response()` | 3. 응답 구조 분석 |
| `system_instruction_demo()` | 4. 시스템 지시문 |
| `token_counting_demo()` | 5. 토큰 사전 카운팅 |
| `error_handling_demo()` | 6. 에러 처리 |
| `list_models()` | 7. 사용 가능한 모델 목록 조회 |

노트북으로 나누면 학생들이 어느 요청에서 어떤 결과가 나오는지 더 쉽게 추적할 수 있습니다.

## 9. 연습 문제

아래 중 하나를 골라 직접 셀을 추가해 보세요.

1. 자기소개 문장을 3가지 톤으로 다시 작성하기
2. FastAPI 개념을 초등학생에게 설명하기
3. 같은 질문에 `max_output_tokens` 값을 바꿔 보기
4. 일부러 긴 입력을 만들고 토큰 수 비교하기

In [ ]:
# 연습용 셀입니다. 아래 프롬프트를 바꿔 실행해 보세요.

practice_prompt = "FastAPI를 처음 배우는 학생에게 한 문단으로 설명해주세요."

# response = call_gemini(
#     "연습 문제",
#     lambda: client.models.generate_content(
#         model=MODEL,
#         contents=practice_prompt,
#         config=types.GenerateContentConfig(max_output_tokens=200),
#     ),
# )
# show_response("연습 답변", response)